initial set-up

In [2]:
''' caio '''

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from datetime import timedelta

df = pd.read_csv('Dati_progetto.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m%d%Y %H:%M')
df = df.sort_values('timestamp').reset_index(drop=True)

df['date'] = df['timestamp'].dt.normalize()
df['hour'] = df['timestamp'].dt.hour + 1     # 1..24
df['dow']  = df['timestamp'].dt.dayofweek  
print(df)

       ZONEID           timestamp  Forecasted Total Load  \
0           1 2011-01-01 00:00:00                  15187   
1           1 2011-01-01 01:00:00                  14464   
2           1 2011-01-01 02:00:00                  13940   
3           1 2011-01-01 03:00:00                  13609   
4           1 2011-01-01 04:00:00                  13391   
...       ...                 ...                    ...   
25963       1 2013-12-17 19:00:00                  23091   
25964       1 2013-12-17 20:00:00                  22504   
25965       1 2013-12-17 21:00:00                  21538   
25966       1 2013-12-17 22:00:00                  20025   
25967       1 2013-12-17 23:00:00                  18306   

       Forecasted Zonal Load  Zonal Price       date  hour  dow  
0                       5091        43.17 2011-01-01     1    5  
1                       4918        36.24 2011-01-01     2    5  
2                       4763        34.64 2011-01-01     3    5  
3              

matrix day x hour

In [3]:
prices = df.pivot_table(index = 'date', columns = 'hour', values = 'Zonal Price', ).sort_index()
loads = df.pivot_table(index = 'date', columns = 'hour', values = 'Forecasted Zonal Load')


logaritmichal tranformation (staibilized the variance)

In [4]:
log_p = np.log(prices)
log_load = np.log(loads)

remove the Nan values of the price column

In [5]:
valid = log_p.dropna().index
log_p, log_load = log_p.loc[valid], log_load.loc[valid]

ARX function
 pd,h = βh,0 + βh,1pd−1,h + βh,2pd−2,h + βh,3pd−7,h + βh,4pmin + βh,5zt + i∈{1,6,7} βh,i+5Di + εd,h

In [6]:
def build_training(target_date, h, window_start):
    X, y = [], []
    d = window_start + timedelta(days=7)
    while d < target_date:
        try: 
            X.append([
                log_p.loc[d - timedelta(days = 1), h], 
                log_p.loc[d - timedelta(days=2), h],
                log_p.loc[d - timedelta(days=7), h],
                log_p.loc[d - timedelta(days=1)].min(),
                log_load.loc[d, h],
                *[1.0 if d.dayofweek == k else 0.0 for k in range(6)]
            ])
            y.append(log_p.loc[d,h])
        except KeyError:
            pass
        d += timedelta(days = 1)
    return np.array(X), np.array(y)

In [7]:
def predict_arx(model, target_date, h):
    X = np.array([[
        log_p.loc[target_date - timedelta(days=1), h],
        log_p.loc[target_date - timedelta(days=2), h],
        log_p.loc[target_date - timedelta(days=7), h],
        log_p.loc[target_date - timedelta(days=1)].min(),
        log_load.loc[target_date, h],
        *[1.0 if target_date.dayofweek == k else 0.0 for k in range(6)]
    ]])
    return model.predict(X)[0]